In [0]:
%pip install jira

In [0]:
# ===============================================

# Jira → Bronze Landing (User-Scoped)

# ===============================================

from jira import JIRA

from pyspark.sql.types import StructType, StructField, StringType

from pyspark.sql import functions as F

from pyspark.sql import Window

from datetime import datetime, timezone

import re

# ----------------------------

# 1️⃣ Parameters

# ----------------------------

user_id     = dbutils.widgets.get("user_id")
project_key = dbutils.widgets.get("project_key")
source      = dbutils.widgets.get("source")

BRONZE_TABLE = f"workspace.slainte_bronze.{source}_issues_bronze"

# ----------------------------

# 2️⃣ Credentials

# ----------------------------

creds = (

    spark.table("workspace.slainte_bronze.jira_integrations_bronze")

    .filter((F.col("user_id") == user_id) & (F.col("project_key") == project_key))

    .orderBy(F.col("ingestion_timestamp").desc())

    .limit(1)

    .collect()[0]

)

site_url = str(creds["site_url"]).strip()

JIRA_URL = site_url if site_url.startswith("http") else f"https://{site_url}"

EMAIL = creds["user_email"]

API_TOKEN = creds["api_token"]

integration_id = creds["id"]

print(f"🔌 Connecting to {JIRA_URL}")

# ----------------------------

# 3️⃣ Mapping

# ----------------------------

def clean(name):

    return re.sub(r'[^\w]', '_', name.lower())

mappings_df = spark.table("workspace.slainte_bronze.supabase_field_mappings_raw")\
    .filter((F.col("user_id") == user_id) & (F.col("project_key") == project_key))

raw_mappings = {r["target_field_name"]: r["source_field_name"] for r in mappings_df.collect()}

mappings = {k: clean(v) for k, v in raw_mappings.items()}

print("📐 mappings:", mappings)

# ----------------------------

# 4️⃣ Jira Connect

# ----------------------------

jira = JIRA(server=JIRA_URL, basic_auth=(EMAIL, API_TOKEN))

# ----------------------------

# 🔥 SLA PARSER

# ----------------------------

def extract_sla(sla_field):

    if not sla_field:

        return None, None

    try:

        if hasattr(sla_field, "completedCycles") and sla_field.completedCycles:

            cycle = sla_field.completedCycles[0]

            return (

                str(cycle.elapsedTime.millis),

                str(cycle.breached)

            )

        if hasattr(sla_field, "ongoingCycle") and sla_field.ongoingCycle:

            cycle = sla_field.ongoingCycle

            return (

                str(cycle.elapsedTime.millis),

                str(cycle.breached)

            )

    except Exception:

        return None, None

    return None, None

# ----------------------------

# 🔥 FIXED STOP TIME EXTRACTION

# ----------------------------

def extract_sla_stop_time(sla_field):

    if not sla_field:

        return None

    try:

        if hasattr(sla_field, "completedCycles") and sla_field.completedCycles:

            cycle = sla_field.completedCycles[0]

            if hasattr(cycle, "stopTime") and cycle.stopTime:

                return str(cycle.stopTime.iso8601)   # ✅ FIX HERE

    except Exception:

        return None

    return None

# ----------------------------

# 5️⃣ Extract

# ----------------------------

issues = jira.search_issues(f'project = {project_key}', maxResults=False)

rows = []

for issue in issues:

    f = issue.fields

    fr_time, fr_breach = extract_sla(getattr(f, "customfield_10118", None))

    res_time, res_breach = extract_sla(getattr(f, "customfield_10117", None))

    # ✅ FIXED resolved_at

    resolved_sla = extract_sla_stop_time(getattr(f, "customfield_10117", None))

    status = str(getattr(f.status, "name", None)).lower()

    if resolved_sla:

        resolved_at = resolved_sla

    elif status in ["done", "closed", "resolved"]:

        resolved_at = str(f.updated)

    else:

        resolved_at = None

    row = {

        "user_id": str(user_id),

        "integration_id": str(integration_id),

        "project_key": str(project_key),

        "ingestion_timestamp": datetime.now(timezone.utc).isoformat(),

        "ticket_number": str(issue.key),

        "ticket_type": str(getattr(f.issuetype, "name", None)),

        "priority": str(getattr(f.priority, "name", None)),

        "status": str(getattr(f.status, "name", None)),

        "assignee": str(getattr(f.assignee, "displayName", None)),

        "created_at": str(f.created),

        "last_update": str(f.updated),

        # ✅ FINAL CORRECT VALUE

        "resolved_at": str(resolved_at),

        "due_date": str(getattr(f, "duedate", None)),

        "support_group": ",".join(f.labels) if f.labels else None,

        "first_response_time_ms": str(fr_time),

        "first_response_breached": str(fr_breach),

        "resolution_time_ms": str(res_time),

        "resolution_breached": str(res_breach),

    }

    rows.append(row)

# ----------------------------

# 6️⃣ Schema

# ----------------------------

schema = StructType([

    StructField("user_id", StringType(), True),

    StructField("integration_id", StringType(), True),

    StructField("project_key", StringType(), True),

    StructField("ingestion_timestamp", StringType(), True),

    StructField("ticket_number", StringType(), True),

    StructField("ticket_type", StringType(), True),

    StructField("priority", StringType(), True),

    StructField("status", StringType(), True),

    StructField("assignee", StringType(), True),

    StructField("created_at", StringType(), True),

    StructField("last_update", StringType(), True),

    StructField("resolved_at", StringType(), True),

    StructField("due_date", StringType(), True),

    StructField("support_group", StringType(), True),

    StructField("first_response_time_ms", StringType(), True),

    StructField("first_response_breached", StringType(), True),

    StructField("resolution_time_ms", StringType(), True),

    StructField("resolution_breached", StringType(), True),

])

df = spark.createDataFrame(rows, schema)

# ----------------------------

# 7️⃣ Dedup

# ----------------------------

df = df.withColumn("_updated_ts", F.to_timestamp("last_update"))

w = Window.partitionBy("user_id", "project_key", "ticket_number")\
          .orderBy(F.col("_updated_ts").desc())

df = df.withColumn("rn", F.row_number().over(w))\
       .filter("rn=1")\
       .drop("rn", "_updated_ts")

# ----------------------------

# 8️⃣ Priority Label

# ----------------------------

pr = F.lower(F.col("priority"))

df = df.withColumn(

    "priority_label",

    F.when(pr.contains("high"), 1)

     .when(pr.contains("medium"), 2)

     .when(pr.contains("low"), 3)

     .otherwise(99)

)

# ----------------------------

# 9️⃣ Write Bronze

# ----------------------------

if not spark.catalog.tableExists(BRONZE_TABLE):

    df.write.mode("overwrite").format("delta").saveAsTable(BRONZE_TABLE)

else:

    df.write.mode("append").format("delta").saveAsTable(BRONZE_TABLE)

print("✅ Bronze DONE")
 